<a href="https://colab.research.google.com/github/Kauustubbh/Skin-Lesion/blob/main/notebooks/01_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import userdata

# Authenticate with Kaggle API using Colab Secrets
os.environ['Kaggle_api_key'] = userdata.get('Kaggle_api_key')

# Download the HAM10000 dataset zip file
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000

# Unzip silently into local /content/data/
!unzip -q skin-cancer-mnist-ham10000.zip -d data/

# Verify the unzipped files exist
!ls data/

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Set and create the persistent project folder
PROJECT_DIR = '/content/drive/MyDrive/SkinLesionClassifier'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"Project directory ready at: {PROJECT_DIR}")

In [ ]:
import glob
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Set visualization styles
sns.set_theme(style='whitegrid')
plt.rcParams['figure.autolayout'] = True

In [ ]:
df = pd.read_csv('data/HAM10000_metadata.csv')

print(f"Dataset shape: {df.shape}")  # Must show exactly (10015, 7)
df.head()

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
dx_names = {
    'akiec': 'Actinic keratoses',
    'bcc':   'Basal cell carcinoma',
    'bkl':   'Benign keratosis-like lesions',
    'df':    'Dermatofibroma',
    'mel':   'Melanoma',
    'nv':    'Melanocytic nevi',
    'vasc':  'Vascular lesions'
}

df['dx_full'] = df['dx'].map(dx_names)
df[['lesion_id', 'image_id', 'dx', 'dx_full']].head()

## Visualize Class Distribution (Imbalance Analysis)

In [ ]:
plt.figure(figsize=(10, 5))
order = df['dx_full'].value_counts().index
sns.countplot(data=df, y='dx_full', order=order, palette='viridis')

plt.title('HAM10000 Class Distribution', fontsize=14, pad=12)
plt.xlabel('Number of Images', fontsize=12)
plt.ylabel('')
plt.show()

# Print exact percentages
percentages = df['dx'].value_counts(normalize=True) * 100
counts = df['dx'].value_counts()
dist_summary = pd.DataFrame({'Count': counts, 'Percentage': percentages.round(2)})
print(dist_summary)

## Build Image Path Index and Plot Sample Grid

In [ ]:
# HAM10000 stores images split across two separate subfolders
img_paths = glob.glob('data/HAM10000_images_part_1/*.jpg') + glob.glob('data/HAM10000_images_part_2/*.jpg')

# Map each image_id (filename without .jpg) to its absolute file path
img_lookup = {os.path.splitext(os.path.basename(p))[0]: p for p in img_paths}

print(f"Total images found: {len(img_lookup)}")  # Must be exactly 10015
assert len(img_lookup) == 10015, f"Expected 10015 images, but found {len(img_lookup)}."

# Display one sample image per class
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, dx_code in enumerate(order):
    sample_row = df[df['dx_full'] == dx_code].iloc[0]
    sample_img_id = sample_row['image_id']
    img_path = img_lookup[sample_img_id]

    img = Image.open(img_path)
    axes[i].imshow(img)
    axes[i].set_title(f"{dx_code}\n({sample_img_id})", fontsize=11)
    axes[i].axis('off')

# Hide unused subplot (8 slots for 7 classes)
axes[7].axis('off')
plt.show()

## Cell 9: Demonstrate Why Splitting by lesion_id is Mandatory
What it does: Evaluates unique lesion counts against total rows and displays instances where a single lesion was captured multiple times.

Why it matters: HAM10000 contains 7,470 unique lesions across 10,015 images. Splitting by image_id would result in near-identical photos of the same physical lesion landing in both the training set and the test set (data leakage). That would artificially inflate validation/test performance and invalidate the evaluation.

In [ ]:
num_unique_lesions = df['lesion_id'].nunique()
total_images = len(df)

print(f"Unique lesion IDs: {num_unique_lesions}")
print(f"Total image files: {total_images}")
print(f"Repeated lesion photos: {total_images - num_unique_lesions}\n")

# Display examples of lesions photographed multiple times
multi_image_lesions = df['lesion_id'].value_counts()
print("Top 5 lesions with multiple captures:")
print(multi_image_lesions.head(5))

## Cell 10: Perform Lesion-Level Stratified Train/Val/Test Split
What it does:Reduces the DataFrame to unique lesion_id entries.
Runs a 70/15/15 stratified partition on dx.
Maps those partitions back to all rows in df.
Why it matters: Guarantees that every image belonging to a particular physical lesion is confined to a single partition, while preserving identical class percentage ratios across all sets.   
Potential gotchas: Leaving out stratify=lesion_df['dx'] will cause rare classes like vasc or df to be distributed unevenly across splits.

In [ ]:
# 1. Deduplicate by lesion_id so each physical lesion appears exactly once
lesion_df = df.drop_duplicates(subset='lesion_id')[['lesion_id', 'dx']]

RANDOM_STATE = 42  # LOCKED: Do not alter to maintain downstream reproducibility

# 2. First split: 70% Train, 30% Temporary (Validation + Test)
train_lesions, temp_lesions = train_test_split(
    lesion_df,
    test_size=0.30,
    stratify=lesion_df['dx'],
    random_state=RANDOM_STATE
)

# 3. Second split: Split the 30% temp equally into 15% Val and 15% Test
val_lesions, test_lesions = train_test_split(
    temp_lesions,
    test_size=0.50,
    stratify=temp_lesions['dx'],
    random_state=RANDOM_STATE
)

# 4. Map the assigned split back to the main image dataframe
split_map = {}
split_map.update({lid: 'train' for lid in train_lesions['lesion_id']})
split_map.update({lid: 'val'   for lid in val_lesions['lesion_id']})
split_map.update({lid: 'test'  for lid in test_lesions['lesion_id']})

df['split'] = df['lesion_id'].map(split_map)
print("Splitting complete. Main DataFrame updated with 'split' column.")

## Cell 11: Validate Split Integrity and Check for Data Leakage
What it does: Computes value distributions for verification, presents the percentage of each disease class within train, val, and test, and programmatically asserts that no lesion_id appears in more than one partition.

Why it matters: If assert leaked_lesions == 0 fails, downstream training will be compromised. This test catches logic or indexing bugs before files are exported.

In [ ]:
print("--- Image Count per Split ---")
print(df['split'].value_counts())
print("\n--- Split Proportions ---")
print(df['split'].value_counts(normalize=True).round(3))

print("\n--- Class Percentage Distribution Across Splits ---")
distribution_check = df.groupby('split')['dx'].value_counts(normalize=True).unstack().T * 100
print(distribution_check.round(2))

# Critical Leakage Assertion Test
lesions_per_split = df.groupby('lesion_id')['split'].nunique()
leaked_lesions = (lesions_per_split > 1).sum()

assert leaked_lesions == 0, f"CRITICAL LEAK DETECTED: {leaked_lesions} lesions cross multiple splits!"
print("\nPASSED: Zero data leakage confirmed. Every lesion_id exists in exactly one split.")

## Cell 12: Export the Split Contract CSV to Google Drive
What it does: Exports the minimal partition metadata (image_id, lesion_id, dx, split) to the shared Google Drive folder and reads it back to confirm disk persistence.   Why it matters: This file is the primary data contract for the project. Notebooks 02_training.ipynb and 03_evaluation.ipynb rely directly on this CSV.

In [ ]:
# Select only the columns needed by downstream notebooks
handoff_df = df[['image_id', 'lesion_id', 'dx', 'split']]

output_path = os.path.join(PROJECT_DIR, 'train_val_test_split.csv')
handoff_df.to_csv(output_path, index=False)

print(f"Handoff CSV successfully saved to:\n{output_path}")

# Sanity check: Read the saved file directly from Drive
test_read = pd.read_csv(output_path)
print(f"Verified saved file on Drive. Rows: {len(test_read)}, Columns: {list(test_read.columns)}")

## Next Notebook guide: Data Augmentation Specification

In [ ]:
"""
Data Augmentation Plan (Reference for 02_training.ipynb)
------------------------------------------------------
The training pipeline in 02_training.ipynb should implement the following
PyTorch transform strategies based on clinical and structural constraints:

1. Training Pipeline (Train Split ONLY):
   - transforms.RandomResizedCrop(224, scale=(0.8, 1.0))
   - transforms.RandomHorizontalFlip(p=0.5)
   - transforms.RandomVerticalFlip(p=0.5)
   - transforms.RandomRotation(degrees=90)
   - transforms.ToTensor()
   - transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

2. Validation & Test Pipeline (Val and Test Splits):
   - transforms.Resize(256)
   - transforms.CenterCrop(224)
   - transforms.ToTensor()
   - transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

Clinical Justification:
- Dermoscopic images have no inherent anatomical 'up' or 'down'; rotation and reflections
  are anatomically invariant.
- Aggressive color jittering and hue adjustments are STRICTLY PROHIBITED to prevent
  distorting clinical color variegation signals (the 'C' in the dermatological ABCDE rules).
"""
print("Augmentation plan documented for 02_training.ipynb.")